# POST PROCESSING

This notebook turns the tensors retrieved from the FPGA into object detections and draws them on the original
image.

It uses only NumPy and Pillow, with no YOLO or Ultralytics module. Decoding and non-maximum suppression follow
YOLOv5 v7.0 with the default `detect.py` settings, so the result can be compared fairly with a reference model.

**Files read** (none of them is changed or deleted):

| File | From | Used for |
|---|---|---|
| `status.txt` | `retrieving_tensor.tcl` | Checking that the run finished without error, and the latency |
| `det0.txt`, `det1.txt`, `det2.txt` | `retrieving_tensor.tcl` | The three Detect output tensors |
| `frame_meta.json` | `preparation.ipynb` | The original image path, size, scale ratio, and padding |
| `frame_load.txt` | `preparation.ipynb` | Its timestamp only, to warn when the results are older than the prepared image |
| `params/yolov5s_meta.json` | this repository | Anchors, strides, and the 80 class names |
| `images/<image>` | you | The original image, as recorded in `frame_meta.json` |

**Files written** to `results/`: `<image>_result.jpg` (image with boxes) and `<image>_detections.txt`
(one detection per line).

In [ ]:
import json
import os

import numpy as np
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont, ImageOps

CONF = 0.25      # confidence threshold
IOU = 0.45       # NMS IoU threshold
MAX_DET = 1000   # maximum detections per image

META_MODEL = json.load(open(os.path.join("params", "yolov5s_meta.json")))
META_FRAME = json.load(open("frame_meta.json"))
RESULT_DIR = "results"

## READ TCL RESULTS

Each line of the files written by the Tcl scripts is an address followed by 32-bit words in hexadecimal.
`read_words` joins them into one byte string and checks that no part is missing.

In [ ]:
def read_words(path):
    """Return the file content as bytes, in address order."""
    chunks, expected = [], None
    for line in open(path):
        t = line.split()
        addr = int(t[0], 16)
        if expected is not None and addr != expected:
            raise ValueError(f"{path} has a gap: run retrieving_tensor.tcl again")
        words = np.array([int(x, 16) for x in t[1:]], dtype="<u4")
        chunks.append(words.tobytes())
        expected = addr + 4 * len(words)
    return b"".join(chunks)

In [ ]:
def check_status(path):
    """Check that the run finished without error. Return (latency in ms, number of layers)."""
    st = np.frombuffer(read_words(path), dtype="<u4")
    if st[0] != 0xC0DE0001:
        raise ValueError("status block not found: run retrieving_tensor.tcl after pressing BTN0")
    if (st[1] >> 1) & 1:
        raise ValueError(f"the run stopped with error code {st[2]}")
    return st[4] / 1e5, int(st[5] & 0xFFFF)

In [ ]:
latency_ms, n_layers = check_status("status.txt")
print(f"inference latency {latency_ms:.2f} ms at 100 MHz, {n_layers} layers")
if os.path.getmtime("status.txt") < os.path.getmtime("frame_load.txt"):
    print("WARNING: these results are older than the last prepared image (frame_load.txt)")

## INTO TENSOR NUMPY

The three Detect outputs are 16-bit fixed point (Q6.10). Only the 255 model channels and the valid width are
kept, then the values are converted to float.

In [ ]:
def to_tensor(raw, h):
    """Return a 255 x h x h float tensor (Detect logits)."""
    t = np.frombuffer(raw, dtype="<i2").reshape(256, h, -1)
    return t[:255, :, :h].astype(np.float64) / 1024.0

In [ ]:
SIZES = [80, 40, 20]
DET = [to_tensor(read_words(f"det{i}.txt"), h) for i, h in enumerate(SIZES)]
for i, t in enumerate(DET):
    print(f"det{i}: {t.shape}, logits {t.min():.3f}..{t.max():.3f}")

## DECODE AND NMS

Each of the 255 channels holds 3 anchors × (x, y, w, h, objectness, 80 class scores). After a sigmoid:

- box center = (2σ − 0.5 + grid cell) × stride
- box size = (2σ)² × anchor
- score = objectness × best class score

Candidates need objectness > `CONF` and score > `CONF`. NMS is done per class by shifting each class's boxes
far apart (as YOLOv5 does), keeping a box only if its IoU with every higher-scoring box is ≤ `IOU`.

In [ ]:
def decode(t, stride, anchors, conf):
    """Return boxes (x1, y1, x2, y2 in 640 x 640 letterbox pixels), scores, classes."""
    h = t.shape[1]
    s = 1.0 / (1.0 + np.exp(-t.reshape(3, 85, h, h)))
    anc = np.array(anchors) * stride
    gy, gx = np.meshgrid(np.arange(h), np.arange(h), indexing="ij")
    cx = (2 * s[:, 0] - 0.5 + gx) * stride
    cy = (2 * s[:, 1] - 0.5 + gy) * stride
    bw = (2 * s[:, 2]) ** 2 * anc[:, 0, None, None]
    bh = (2 * s[:, 3]) ** 2 * anc[:, 1, None, None]
    obj = s[:, 4]
    cls = s[:, 5:].argmax(1)
    score = obj * s[:, 5:].max(1)
    keep = (obj > conf) & (score > conf)
    boxes = np.stack([cx - bw / 2, cy - bh / 2, cx + bw / 2, cy + bh / 2], -1)[keep]
    return boxes, score[keep], cls[keep]

In [ ]:
def nms(boxes, scores, classes, iou, max_det, max_wh=7680, max_nms=30000):
    """Return indices of kept boxes, highest score first."""
    order = np.argsort(-scores, kind="stable")[:max_nms]
    b = boxes[order] + classes[order, None] * float(max_wh)
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    alive = np.ones(len(order), dtype=bool)
    keep = []
    for i in range(len(order)):
        if not alive[i]:
            continue
        keep.append(order[i])
        if len(keep) == max_det:
            break
        rest = np.nonzero(alive[i + 1:])[0] + i + 1
        iw = np.clip(np.minimum(b[i, 2], b[rest, 2]) - np.maximum(b[i, 0], b[rest, 0]), 0, None)
        ih = np.clip(np.minimum(b[i, 3], b[rest, 3]) - np.maximum(b[i, 1], b[rest, 1]), 0, None)
        inter = iw * ih
        alive[rest[inter / (area[i] + area[rest] - inter) > iou]] = False
    return np.array(keep, dtype=int)

In [ ]:
parts = [decode(t, META_MODEL["strides"][i], META_MODEL["anchors_grid_units"][i], CONF) for i, t in enumerate(DET)]
boxes = np.concatenate([p[0] for p in parts])
scores = np.concatenate([p[1] for p in parts])
classes = np.concatenate([p[2] for p in parts])
keep = nms(boxes, scores, classes, IOU, MAX_DET)
boxes, scores, classes = boxes[keep], scores[keep], classes[keep]
print(f"{len(parts[0][0]) + len(parts[1][0]) + len(parts[2][0])} candidates, {len(keep)} detections after NMS")

## MAPPING TO THE IMAGE

Boxes are moved from the 640×640 letterbox back to the original image: remove the padding, divide by the scale
ratio, and clip to the image border. As in YOLOv5, the padding is recomputed from the image size
(it can be fractional). The original image is the one recorded in `frame_meta.json`, so the boxes always match
the image that was sent.

In [ ]:
def scale_boxes(boxes, orig_w, orig_h, size=640):
    """Map letterbox boxes to original image coordinates."""
    gain = min(size / orig_h, size / orig_w)
    pad_x = (size - orig_w * gain) / 2
    pad_y = (size - orig_h * gain) / 2
    b = boxes.copy()
    b[:, [0, 2]] = ((b[:, [0, 2]] - pad_x) / gain).clip(0, orig_w)
    b[:, [1, 3]] = ((b[:, [1, 3]] - pad_y) / gain).clip(0, orig_h)
    return b

In [ ]:
ORIGINAL_IMAGE_PATH = META_FRAME["source"]
boxes_img = scale_boxes(boxes, META_FRAME["orig_w"], META_FRAME["orig_h"])
print(f"original image: {ORIGINAL_IMAGE_PATH} ({META_FRAME['orig_w']} x {META_FRAME['orig_h']})")

## PREVIEW

In [ ]:
def draw_boxes(img, boxes, scores, classes, names):
    """Return a copy of img with boxes and labels drawn."""
    img = img.copy()
    dr = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", max(14, img.size[0] // 40))
    except OSError:
        font = ImageFont.load_default()
    line = max(2, img.size[0] // 250)
    for (x1, y1, x2, y2), sc, c in zip(boxes, scores, classes):
        label = f"{names[int(c)]} {sc:.2f}"
        col = tuple(int(v) for v in np.array([37, 97, 173]) * (int(c) + 3) % 255)
        dr.rectangle([x1, y1, x2, y2], outline=col, width=line)
        tw, th = dr.textbbox((0, 0), label, font=font)[2:]
        ty = max(0, y1 - th - 4)
        dr.rectangle([x1, ty, x1 + tw + 6, ty + th + 4], fill=col)
        dr.text((x1 + 3, ty + 2), label, fill=(255, 255, 255), font=font)
    return img

In [ ]:
names = META_MODEL["names"]
original = ImageOps.exif_transpose(Image.open(ORIGINAL_IMAGE_PATH)).convert("RGB")
result = draw_boxes(original, boxes_img, scores, classes, names)
for (x1, y1, x2, y2), sc, c in zip(boxes_img, scores, classes):
    print(f"{names[int(c)]:<14} {sc:.3f}  [{x1:6.1f} {y1:6.1f} {x2:6.1f} {y2:6.1f}]")
display(result)

## SAVE IMAGE

Saves the image with boxes and a text file of the detections (one per line: class id, class name, score,
x1, y1, x2, y2 in original image pixels).

In [ ]:
def save_results(img, boxes, scores, classes, names, name, out_dir=RESULT_DIR):
    """Save <name>_result.jpg and <name>_detections.txt. Return both paths."""
    os.makedirs(out_dir, exist_ok=True)
    img_path = os.path.join(out_dir, f"{name}_result.jpg")
    txt_path = os.path.join(out_dir, f"{name}_detections.txt")
    img.save(img_path, quality=95)
    with open(txt_path, "w") as f:
        f.write("# class_id class_name score x1 y1 x2 y2\n")
        for (x1, y1, x2, y2), sc, c in zip(boxes, scores, classes):
            f.write(f"{int(c)} {names[int(c)].replace(' ', '_')} {sc:.6f} {x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f}\n")
    return img_path, txt_path

In [ ]:
name = os.path.splitext(os.path.basename(ORIGINAL_IMAGE_PATH))[0]
for p in save_results(result, boxes_img, scores, classes, names, name):
    print(f"saved {p}")